# Eurex API FAQ & Special Cases

[![Open In Colab](https://img.shields.io/badge/Google%20Colab-F9AB00?style=for-the-badge&logo=googlecolab&logoColor=white)](https://colab.research.google.com/github/ViktorHD/eurex-api/blob/main/notebooks/FAQ_cases.ipynb) 
[![Binder](https://img.shields.io/badge/Binder-579ACA?style=for-the-badge&logo=binder&logoColor=white)](https://mybinder.org/v2/gh/ViktorHD/eurex-api/main?labpath=notebooks%2FFAQ_cases.ipynb) 
[![Databricks](https://img.shields.io/badge/Databricks-FF3621?style=for-the-badge&logo=databricks&logoColor=white)](https://community.cloud.databricks.com/?o=0#external-import?github_url=https%3A%2F%2Fgithub.com%2FViktorHD%2Feurex-api%2Fblob%2Fmain%2Fnotebooks%2FFAQ_cases.ipynb)

This notebook covers concrete, highly requested examples and recipes for processing reference data from the [Eurex GraphQL API](https://www.eurex.com/ex-en/data/free-reference-data-api).

## Case 1: Breaking Down Product-Level MinLotSize to Single Contract Level

### Problem Description
Within the Eurex reference data model, block trade thresholds (`MinLotSize`) and non-disclosure limits from `TESProfiles` are defined on the **product** level, depending on whether the instrument is standard or flexible. For standard/simple instruments, these thresholds can vary based on how far out the expiry date is.

Specifically, `TESProfiles` has a `MinExpiryRange` field, which indicates the **starting point** of an expiration range. A particular `MinLotSize` and `NonDisclosureLimit` apply from that `MinExpiryRange` value onwards until the next threshold is reached. `MinExpiryRange` represents the expiration index (i.e. the chronological sequence of contract expirations, starting from `1` for the front month/closest expiry).

**Example:** If a product has `MinExpiryRange` values of `[1, 11]` with corresponding `MinLotSize` values of `[1500, 1]`, this means:
* ExpirationIndex 1-10 → MinLotSize = 1500
* ExpirationIndex 11+ → MinLotSize = 1

To determine the actual thresholds for a **single individual contract** (at the contract ID/ISIN level), we need to:
1. **Fetch Expirations:** For a given product, query `Expirations` to retrieve the mapping between a `MasterContract` and its chronological `ExpirationIndex` (along with `ProductID` and `ExpirationDate`).
2. **Fetch TESProfiles:** Query `TESProfiles` to get the `MinLotSize`, `NonDisclosureLimit`, and `MinExpiryRange` thresholds for standard simple instruments (`InstrumentType: "SIMPLE_INSTRUMENT"` and `TESType: "BLOCK"`).
3. **Fetch Contracts:** Query `Contracts` to obtain individual contracts for that product.
4. **Align and Map:** Join the Contracts and Expirations on `ProductID` and `MasterContract` to associate an `ExpirationIndex` with each single contract. Then, map that index to the corresponding thresholds from the product's `TESProfiles` by finding the highest `MinExpiryRange` where `ExpirationIndex >= MinExpiryRange`.

### 1. Setup & API Helper

We start by importing the necessary libraries and establishing a connection to the Eurex GraphQL API using the public demo key.

In [0]:
import sys
!{sys.executable} -m pip install pandas requests
import requests
import json
import pandas as pd

API_URL = "https://api.developer.deutsche-boerse.com/eurex-prod-graphql/"
# Public demo key from: https://www.eurex.com/ex-en/data/free-reference-data-api
API_KEY = "68cdafd2-c5c1-49be-8558-37244ab4f513"

headers = {
    "Content-Type": "application/json",
    "X-DBP-APIKEY": API_KEY
}

def run_query(query):
    payload = {'query': query}
    response = requests.post(API_URL, json=payload, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Query failed with status code {response.status_code}: {response.text}")

### 2. Breakdown Logic Function

We define a robust, generic Python function `breakdown_min_lot_size(product_code)` which dynamically fetches and joins all data from the API to allocate the proper `MinLotSize` to individual contracts.

In [0]:
def breakdown_min_lot_size(product_code):
    """
    Performs a 3-step retrieval and join to allocate product-level TES block trade thresholds
    (MinLotSize) to individual contracts for a given product filter.
    """
    print(f"Processing MinLotSize breakdown for product: {product_code}...\n")
    
    # Step 1: Fetch Expirations
    exp_query = f"""
    query {{
      Expirations(filter: {{ Product: {{ eq: \"{product_code}\" }} }}) {{
        data {{
          ProductID
          Product
          MasterContract
          ExpirationIndex
          ExpirationDate
        }}
      }}
    }}
    """
    exp_res = run_query(exp_query)
    exp_data = exp_res.get("data", {}).get("Expirations", {}).get("data", [])
    if not exp_data:
        print(f"No expirations found for product {product_code}.")
        return pd.DataFrame()
    df_exp = pd.DataFrame(exp_data)
    
    # Step 2: Fetch TESProfiles for InstrumentType = SIMPLE_INSTRUMENT and TESType = BLOCK
    tes_query = f"""
    query {{
      TESProfiles(filter: {{ 
        Product: {{ eq: \"{product_code}\" }}, 
        InstrumentType: {{ eq: \"SIMPLE_INSTRUMENT\" }}, 
        TESType: {{ eq: \"BLOCK\" }} 
      }}) {{
        data {{ 
          ProductID
          Product
          InstrumentType
          TESType
          MinLotSize
          NonDisclosureLimit
          MinExpiryRange
        }}
      }}
    }}
    """
    tes_res = run_query(tes_query)
    tes_data = tes_res.get("data", {}).get("TESProfiles", {}).get("data", [])
    if not tes_data:
        print(f"No BLOCK TES profiles found for product {product_code}.")
        return pd.DataFrame()
    df_tes = pd.DataFrame(tes_data)
    
    # Step 3: Fetch Contracts
    contracts_query = f"""
    query {{
      Contracts(filter: {{ Product: {{ eq: \"{product_code}\" }} }}) {{
        data {{
          ProductID
          Product
          MasterContract
          ContractID
          Contract
          ExpirationDate
          ISIN
        }}
      }}
    }}
    """
    contracts_res = run_query(contracts_query)
    contracts_data = contracts_res.get("data", {}).get("Contracts", {}).get("data", [])
    if not contracts_data:
        print(f"No contracts found for product {product_code}.")
        return pd.DataFrame()
    df_contracts = pd.DataFrame(contracts_data)
    
    # Align: Join Contracts with Expirations on ProductID and MasterContract
    df_merged = pd.merge(
        df_contracts, 
        df_exp,
        on=["ProductID", "MasterContract"],
        suffixes=("", "_exp")
    )
    
    # Keep relevant contract columns
    df_merged = df_merged[[
        "ProductID", "Product", "MasterContract", "ContractID", 
        "Contract", "ExpirationDate", "ExpirationIndex", "ISIN"
    ]]
    
    # Allocate: Map ExpirationIndex to MinLotSize using sorted TESProfiles
    # MinExpiryRange indicates the START of a range (not the upper limit)
    # Sort descending to find the applicable range from highest to lowest
    df_tes_sorted = df_tes.sort_values("MinExpiryRange", ascending=False).copy()
    
    def find_threshold_value(exp_idx, df_profiles, field_name):
        # Find the first profile where ExpirationIndex >= MinExpiryRange
        # (profiles are sorted descending, so we find the highest applicable threshold)
        valid_profiles = df_profiles[exp_idx >= df_profiles["MinExpiryRange"]]
        if not valid_profiles.empty:
            # Return the field value for the first valid range (highest threshold that applies)
            return valid_profiles.iloc[0][field_name]
        else:
            # Should not happen if data is complete, but fallback to first profile
            return df_profiles.iloc[-1][field_name] if not df_profiles.empty else None
            
    df_merged["MinLotSize"] = df_merged["ExpirationIndex"].apply(
        lambda idx: find_threshold_value(idx, df_tes_sorted, "MinLotSize")
    )
    df_merged["NonDisclosureLimit"] = df_merged["ExpirationIndex"].apply(
        lambda idx: find_threshold_value(idx, df_tes_sorted, "NonDisclosureLimit")
    )
    
    return df_merged

### 3. Products with MinLotSize Differentiation Overview

Before running specific examples, let's get an overview of which products have MinLotSize differentiation based on expiration ranges. We'll query all products and identify those with multiple MinLotSize tiers, grouped by ProductType.

**Important:** Since the breakdown function requires joining with the Expirations table, we also check which products have corresponding Expirations records. Products without Expirations data will be marked as `Has_Expirations = 'No'` and the breakdown function will not work for them.

In [0]:
# Query all TESProfiles with MinLotSize differentiation
tes_overview_query = """
query {
  TESProfiles(filter: { 
    InstrumentType: { eq: "SIMPLE_INSTRUMENT" }, 
    TESType: { eq: "BLOCK" } 
  }) {
    data {
      ProductID
      Product
      MinLotSize
      MinExpiryRange
    }
  }
}
"""

tes_overview_res = run_query(tes_overview_query)
tes_overview_data = tes_overview_res.get("data", {}).get("TESProfiles", {}).get("data", [])
df_tes_overview = pd.DataFrame(tes_overview_data)

# Query ProductInfos to get ProductType
products_query = """
query {
  ProductInfos {
    data {
      ProductID
      Product
      ProductType
    }
  }
}
"""

products_res = run_query(products_query)
products_data = products_res.get("data", {}).get("ProductInfos", {}).get("data", [])
df_products = pd.DataFrame(products_data)

# Query Expirations to check which products have expiration data
expirations_query = """
query {
  Expirations {
    data {
      ProductID
      Product
    }
  }
}
"""

expirations_res = run_query(expirations_query)
expirations_data = expirations_res.get("data", {}).get("Expirations", {}).get("data", [])
df_expirations = pd.DataFrame(expirations_data)

# Get unique products with expirations
products_with_expirations = set(df_expirations['Product'].unique()) if not df_expirations.empty else set()

# Merge to get ProductType
df_tes_with_type = pd.merge(
    df_tes_overview,
    df_products[["ProductID", "ProductType"]],
    on="ProductID",
    how="left"
)

# Group by Product and ProductType to count distinct MinLotSize values
product_summary = df_tes_with_type.groupby(["Product", "ProductType"]).agg(
    MinLotSize_Count=('MinLotSize', 'nunique'),
    MinLotSize_Values=('MinLotSize', lambda x: sorted(x.unique())),
    MinExpiryRange_Values=('MinExpiryRange', lambda x: sorted(x.unique()))
).reset_index()

# Add column to mark products without Expirations data
product_summary['Has_Expirations'] = product_summary['Product'].apply(
    lambda p: 'Yes' if p in products_with_expirations else 'No'
)

# Filter to products with differentiation (more than 1 MinLotSize)
products_with_differentiation = product_summary[product_summary['MinLotSize_Count'] > 1].sort_values(
    ['ProductType', 'Product']
)

print(f"Total products with MinLotSize differentiation: {len(products_with_differentiation)}")
print(f"\nBreakdown by ProductType:")
print(products_with_differentiation.groupby('ProductType').size())

print(f"\nProducts WITHOUT Expirations data (breakdown will not work):")
products_without_expirations = products_with_differentiation[products_with_differentiation['Has_Expirations'] == 'No']
print(f"Count: {len(products_without_expirations)}")
if len(products_without_expirations) > 0:
    print(products_without_expirations[['Product', 'ProductType']].to_string(index=False))

print(f"\n--- Products with MinLotSize Differentiation by Expiration Range ---")
print(f"Legend: Has_Expirations = 'Yes' means breakdown function will work, 'No' means missing Expirations data")
display(products_with_differentiation[['Product', 'ProductType', 'MinLotSize_Count', 'MinLotSize_Values', 'MinExpiryRange_Values', 'Has_Expirations']])

### 4. Execution & Results

Let's execute this logic for specific products. The examples below show products with multiple expirations spanning several years and different `MinLotSize` tiers.

**Note:** Make sure to use products where `Has_Expirations = 'Yes'` from the overview above. Products without Expirations data will not work with the breakdown function.

We will print the final joined result displaying the mapped `MinLotSize` and `NonDisclosureLimit` alongside each individual contract.

In [0]:
# Using OESX as an example (Swiss Market Index Options - has Expirations data)
df_result = breakdown_min_lot_size("OESX")

if not df_result.empty:
    print(f"Total contracts processed: {len(df_result)}")
    
    # Sort contracts chronologically by ExpirationIndex for a clear, structured view
    df_result = df_result.sort_values(["ExpirationIndex", "ContractID"])
    
    # Get one representative contract per ExpirationIndex to show the mapping
    example_records = df_result.drop_duplicates(subset=["ExpirationIndex"])
    
    print("\n--- Example Records showing ExpirationIndex mapping to MinLotSize ---")
    display(example_records[["Product", "ContractID", "Contract", "ExpirationIndex", "MinLotSize", "NonDisclosureLimit", "ISIN"]])
else:
    print("Could not process breakdown.")

## Case 2: Fetching All Trading Hours for Product Groups

### Problem Description
Trading hours can vary by product, contract type, and market segment. The Eurex API provides comprehensive trading hours through the `TradingHours` endpoint, which includes multiple time windows for different trading activities.

**Available fields (discovered via introspection):**
* **StartTES / EndTES** - TES (Trade Entry Services) block trade hours
* **StartContinuousTrading / EndContinuousTrading** - Continuous trading phase
* **EndOpeningAuction** - Opening auction end time
* **EndClosingAuction** - Closing auction end time
* **LTDBook** - Last Trading Day for order book
* **LTDTES** - Last Trading Day for TES
* **Product** - Product code
* **ProductID** - Unique product identifier

**Note:** The `TradingHours` endpoint can only be filtered by `Product` or `ProductID`, not by `ProductType`. Therefore, to get trading hours for a product group (e.g., Single Stock Options), we need to:
1. First query `ProductInfos` to get all products of that type
2. Then query `TradingHours` for each product individually or in batches

This example demonstrates the comprehensive approach for Single Stock Options, fetching ALL available time windows.

In [0]:
# Introspect the TradingHours type to discover all available fields
introspection_query = """
query {
  __type(name: "TradingHours") {
    name
    fields {
      name
      type {
        name
        kind
        ofType {
          name
          kind
        }
      }
    }
  }
}
"""

print("Introspecting TradingHours schema...\n")
introspect_res = run_query(introspection_query)
trading_hours_type = introspect_res.get("data", {}).get("__type", {})

if trading_hours_type:
    fields = trading_hours_type.get("fields", [])
    print(f"Found {len(fields)} fields in TradingHours:\n")
    
    # Organize fields by category
    id_fields = []
    time_fields = []
    other_fields = []
    
    for field in fields:
        field_name = field.get("name", "")
        field_type = field.get("type", {}).get("name") or field.get("type", {}).get("ofType", {}).get("name", "")
        
        if field_name in ["ProductID", "Product"]:
            id_fields.append(field_name)
        elif "Start" in field_name or "End" in field_name:
            time_fields.append(field_name)
        else:
            other_fields.append(field_name)
    
    print("=== Identification Fields ===")
    for f in id_fields:
        print(f"  - {f}")
    
    print("\n=== Time-related Fields ===")
    for f in sorted(time_fields):
        print(f"  - {f}")
    
    if other_fields:
        print("\n=== Other Fields ===")
        for f in other_fields:
            print(f"  - {f}")
    
    # Store all field names for the next query
    all_field_names = [f.get("name") for f in fields]
    print(f"\n✓ All {len(all_field_names)} fields discovered and ready to query")
else:
    print("Failed to introspect TradingHours type")

In [0]:
# Step 1: Get Single Stock Options products first
print("Step 1: Fetching Single Stock Options products...\n")
products_query = """
query {
  ProductInfos(filter: { 
    ProductType: { eq: "SINGLE STOCK OPTIONS" } 
  }) {
    data {
      ProductID
      Product
      ProductType
    }
  }
}
"""

products_res = run_query(products_query)
products_data = products_res.get("data", {}).get("ProductInfos", {}).get("data", [])
df_sso_products = pd.DataFrame(products_data)

print(f"Found {len(df_sso_products)} Single Stock Options products")

# Step 2: Fetch ALL trading hours in one query (much faster than looping)
print("\nStep 2: Fetching all trading hours in one query...\n")

trading_hours_query = """
query {
  TradingHours {
    data {
      ProductID
      Product
      StartTES
      EndTES
      StartContinuousTrading
      EndContinuousTrading
      EndOpeningAuction
      EndClosingAuction
      LTDBook
      LTDTES
    }
  }
}
"""

th_res = run_query(trading_hours_query)
all_trading_hours = th_res.get("data", {}).get("TradingHours", {}).get("data", [])

print(f"Fetched {len(all_trading_hours)} trading hours records for all products.")

if all_trading_hours:
    df_all_trading_hours = pd.DataFrame(all_trading_hours)
    
    # Filter to only Single Stock Options products
    sso_product_set = set(df_sso_products['Product'])
    df_trading_hours = df_all_trading_hours[df_all_trading_hours['Product'].isin(sso_product_set)].copy()
    
    # Merge with product info to get ProductType
    df_trading_hours = pd.merge(
        df_trading_hours,
        df_sso_products[['Product', 'ProductType']],
        on='Product',
        how='left'
    )
    
    print(f"Total trading hours records found: {len(df_trading_hours)}")
    print(f"Products with trading hours data: {df_trading_hours['Product'].nunique()}")
    
    # Identify which time fields have data (all discovered fields)
    time_columns = ['StartTES', 'EndTES', 'StartContinuousTrading', 'EndContinuousTrading',
                    'EndOpeningAuction', 'EndClosingAuction', 'LTDBook', 'LTDTES']
    
    available_times = {col: df_trading_hours[col].notna().sum() for col in time_columns if col in df_trading_hours.columns}
    
    print("\n--- Available Time Fields (non-null count) ---")
    for field, count in sorted(available_times.items(), key=lambda x: x[1], reverse=True):
        if count > 0:
            print(f"{field}: {count} products ({count/len(df_trading_hours)*100:.1f}%)")
    
    # Show full breakdown per product with ALL time fields
    print("\n--- Complete Trading Hours Breakdown by Product ---")
    df_display = df_trading_hours.sort_values('Product')
    
    # Select only columns that have data
    display_cols = ['Product', 'ProductType'] + [col for col in time_columns if col in df_trading_hours.columns and available_times.get(col, 0) > 0]
    display(df_display[display_cols])
    
    # Show summary statistics for each time field
    print("\n--- Trading Hours Summary Statistics ---")
    
    # Separate time fields and LTD fields for better organization
    regular_time_cols = [c for c in time_columns if not c.startswith('LTD')]
    ltd_cols = [c for c in time_columns if c.startswith('LTD')]
    
    print("\nRegular Trading Times:")
    for col in regular_time_cols:
        if col in df_trading_hours.columns and available_times.get(col, 0) > 0:
            non_null = df_trading_hours[col].dropna()
            if len(non_null) > 0:
                unique_vals = non_null.nunique()
                if col.startswith('Start'):
                    print(f"  {col}: Earliest = {non_null.min()}, Latest = {non_null.max()}, Most Common = {non_null.mode()[0] if len(non_null.mode()) > 0 else 'N/A'} ({unique_vals} unique values)")
                else:
                    print(f"  {col}: Earliest = {non_null.min()}, Latest = {non_null.max()}, Most Common = {non_null.mode()[0] if len(non_null.mode()) > 0 else 'N/A'} ({unique_vals} unique values)")
    
    print("\nLast Trading Day (LTD) Times:")
    for col in ltd_cols:
        if col in df_trading_hours.columns and available_times.get(col, 0) > 0:
            non_null = df_trading_hours[col].dropna()
            if len(non_null) > 0:
                unique_vals = non_null.nunique()
                print(f"  {col}: Earliest = {non_null.min()}, Latest = {non_null.max()}, Most Common = {non_null.mode()[0] if len(non_null.mode()) > 0 else 'N/A'} ({unique_vals} unique values)")
        elif col in df_trading_hours.columns:
            print(f"  {col}: No data available")
    
else:
    print("No trading hours data found for the sample products.")